In [63]:
import pandas as pd
import numpy as np
import re
import os
os.getcwd()



'c:\\Users\\maina\\Documents\\kdd-project'

# Loading the job dataset and preprocessing

In [64]:
# Columns we want to keep are: job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
# We will have zip_code and fips as strings instead of integers because they can have leading zeros, job_id and company_id as strings
# and we dont want to lose that information. Also, we have everything to lowercase to avoid any issues with case sensitivity when we do our analysis later on.

columns_to_keep = ['job_id', 'company_name', 'title', 'description', 'location', 'company_id', 'views', 'skills_desc', 'work_type', 'zip_code', 'fips']

jobs = pd.read_csv('data/postings.csv',
                   usecols=columns_to_keep,
                   dtype={'job_id': str, 'company_id': str, 'zip_code': str, 'fips': str})


text_cols = ['company_name', 'title', 'description', 'location', 'skills_desc', 'work_type']

for col in text_cols:
    jobs[col] = jobs[col].str.lower().str.strip()


In [104]:
jobs.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021,Princeton,NJ
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069,Fort Collins,CO
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061,Cincinnati,OH
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059,New Hyde Park,NY
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057,Burlington,IA


In [66]:
jobs.tail()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
123844,3906267117,lozano smith,title ix/investigations attorney,our walnut creek office is currently seeking a...,"walnut creek, ca",56120.0,1.0,NaN,full_time,94595,06013
123845,3906267126,pinterest,"staff software engineer, ml serving platform",about pinterest:\n\nmillions of people across ...,united states,1124131.0,3.0,NaN,full_time,NaN,NaN
123846,3906267131,eps learning,"account executive, oregon/washington",company overview\n\neps learning is a leading ...,"spokane, wa",90552133.0,3.0,NaN,full_time,99201,53063
123847,3906267195,trelleborg applied technologies,business development manager,the business development manager is a 'hunter'...,"texas, united states",2793699.0,4.0,NaN,full_time,NaN,NaN
123848,3906267224,solugenix,marketing social media specialist,marketing social media specialist - $70k – $75...,"san juan capistrano, ca",43325.0,2.0,NaN,full_time,92675,06059


In [67]:
jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        123849 non-null  str    
 1   company_name  122130 non-null  str    
 2   title         123849 non-null  str    
 3   description   123842 non-null  str    
 4   location      123849 non-null  str    
 5   company_id    122132 non-null  str    
 6   views         122160 non-null  float64
 7   skills_desc   2439 non-null    str    
 8   work_type     123849 non-null  str    
 9   zip_code      102977 non-null  str    
 10  fips          96434 non-null   str    
dtypes: float64(1), str(10)
memory usage: 10.4 MB


In [68]:
# We notice that some of the rows have country as united states , we dont want this. We have the zip_code and fips code which can help us identify the location of the job,
# We will use the zip_code and fips to get a city and state for each job, for rows where location is united states and no zip_code/fips we will drop those rows. 
# Normalize zip/fips as 5-digit strings (keep leading zeros)


# normalize location safely
jobs['location'] = (
    jobs['location']
    .astype(str)
    .str.strip()
    .str.lower()
)

# define "missing zip" robustly
missing_zip = (
    jobs['zip_code'].isna() |
    (jobs['zip_code'].astype(str).str.strip() == '') |
    (jobs['zip_code'].astype(str).str.lower() == 'nan')
)

mask_drop = (
    jobs['location'].eq('united states') &
    missing_zip &
    jobs['fips'].isna()
)

before = len(jobs)
jobs = jobs[~mask_drop]
after = len(jobs)

print(f"Dropped {before - after} rows")



Dropped 8125 rows


In [69]:
jobs.head(20)

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057
5,91700727,downtown raleigh alliance,economic development and planning intern,job summary:the economic development & plannin...,"raleigh, nc",1481176.0,9.0,NaN,internship,27601,37183
7,112576855,NaN,building engineer,summary: due to the pending retirement of our ...,"san francisco, ca",NaN,2.0,NaN,full_time,94101,06075
8,1218575,children's nebraska,respiratory therapist,"at children’s, the region’s only full-service ...","omaha, ne",721189.0,3.0,• requires the ability to communicate effectiv...,full_time,68102,31055
9,2264355,bay west church,worship leader,it is an exciting time to be a part of our chu...,"palm bay, fl",28631247.0,5.0,"knowledge, skills and abilities: 1. proficient...",part_time,32905,12009
10,9615617,"glastender, inc.",inside customer service associate,glastender inc. is a family-owned manufacturer...,"saginaw, mi",1194336.0,4.0,the production supervisor must possess strong ...,full_time,48601,26145


In [70]:
jobs[jobs['location'] == 'united states'][['location', 'zip_code', 'fips']].head(10)

,location,zip_code,fips


In [71]:
zips_raw = pd.read_csv('data/uszips.csv')
zips_raw.columns = zips_raw.columns.str.strip().str.lower()

state_col = next(
    (c for c in ['state', 'state_id', 'state_name'] if c in zips_raw.columns),
    None
)

missing_required = [c for c in ['zip', 'city'] if c not in zips_raw.columns]
if missing_required:
    raise ValueError(f"Missing required columns in uszips.csv: {missing_required}")

if state_col is None:
    zips = zips_raw[['zip', 'city']].copy()
    zips['state'] = pd.NA
else:
    zips = zips_raw[['zip', 'city', state_col]].copy().rename(columns={state_col: 'state'})

zips['zip'] = zips['zip'].astype('string').str.strip().str.zfill(5)
jobs['zip_code'] = jobs['zip_code'].astype('string').str.strip().str.zfill(5)

# make reruns safe in notebook
jobs = jobs.drop(columns=['zip', 'city', 'state'], errors='ignore')

jobs = jobs.merge(
    zips,
    left_on='zip_code',
    right_on='zip',
    how='left'
)


In [72]:
jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 115724 entries, 0 to 115723
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        115724 non-null  str    
 1   company_name  114283 non-null  str    
 2   title         115724 non-null  str    
 3   description   115720 non-null  str    
 4   location      115724 non-null  str    
 5   company_id    114285 non-null  str    
 6   views         114161 non-null  float64
 7   skills_desc   2409 non-null    str    
 8   work_type     115724 non-null  str    
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   str    
 11  zip           99949 non-null   string 
 12  city          99949 non-null   str    
 13  state         99949 non-null   str    
dtypes: float64(1), str(11), string(2)
memory usage: 12.4 MB


In [73]:
missing_city_state = jobs['city'].isna() & jobs['state'].isna()

In [74]:

STATE_MAP = {
    'alabama': 'AL',
    'alaska': 'AK',
    'arizona': 'AZ',
    'arkansas': 'AR',
    'california': 'CA',
    'colorado': 'CO',
    'connecticut': 'CT',
    'delaware': 'DE',
    'florida': 'FL',
    'georgia': 'GA',
    'hawaii': 'HI',
    'idaho': 'ID',
    'illinois': 'IL',
    'indiana': 'IN',
    'iowa': 'IA',
    'kansas': 'KS',
    'kentucky': 'KY',
    'louisiana': 'LA',
    'maine': 'ME',
    'maryland': 'MD',
    'massachusetts': 'MA',
    'michigan': 'MI',
    'minnesota': 'MN',
    'mississippi': 'MS',
    'missouri': 'MO',
    'montana': 'MT',
    'nebraska': 'NE',
    'nevada': 'NV',
    'new hampshire': 'NH',
    'new jersey': 'NJ',
    'new mexico': 'NM',
    'new york': 'NY',
    'north carolina': 'NC',
    'north dakota': 'ND',
    'ohio': 'OH',
    'oklahoma': 'OK',
    'oregon': 'OR',
    'pennsylvania': 'PA',
    'rhode island': 'RI',
    'south carolina': 'SC',
    'south dakota': 'SD',
    'tennessee': 'TN',
    'texas': 'TX',
    'utah': 'UT',
    'vermont': 'VT',
    'virginia': 'VA',
    'washington': 'WA',
    'west virginia': 'WV',
    'wisconsin': 'WI',
    'wyoming': 'WY',
    'district of columbia': 'DC'
}



In [75]:
def extract_state_from_location(loc):
    if not isinstance(loc, str):
        return None

    loc_lower = loc.lower()

    for name, abbr in STATE_MAP.items():
        abbr_lower = abbr.lower()

        # check full state name
        if name in loc_lower:
            return abbr

        # check abbreviation as a word boundary (tx, TX, Tx, etc.)
        pattern = rf'\b{abbr_lower}\b'
        if re.search(pattern, loc_lower):
            return abbr

    return None



jobs.loc[missing_city_state, 'state'] = (
    jobs.loc[missing_city_state, 'location']
    .apply(extract_state_from_location)
)

jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 115724 entries, 0 to 115723
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        115724 non-null  str    
 1   company_name  114283 non-null  str    
 2   title         115724 non-null  str    
 3   description   115720 non-null  str    
 4   location      115724 non-null  str    
 5   company_id    114285 non-null  str    
 6   views         114161 non-null  float64
 7   skills_desc   2409 non-null    str    
 8   work_type     115724 non-null  str    
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   str    
 11  zip           99949 non-null   string 
 12  city          99949 non-null   str    
 13  state         111748 non-null  str    
dtypes: float64(1), str(11), string(2)
memory usage: 12.4 MB


In [76]:
jobs = jobs.drop(columns=['zip'], errors='ignore')

In [77]:
jobs.head(50)

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021,Princeton,NJ
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069,Fort Collins,CO
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061,Cincinnati,OH
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059,New Hyde Park,NY
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057,Burlington,IA
5,91700727,downtown raleigh alliance,economic development and planning intern,job summary:the economic development & plannin...,"raleigh, nc",1481176.0,9.0,NaN,internship,27601,37183,Raleigh,NC
6,112576855,NaN,building engineer,summary: due to the pending retirement of our ...,"san francisco, ca",NaN,2.0,NaN,full_time,94101,06075,NaN,CA
7,1218575,children's nebraska,respiratory therapist,"at children’s, the region’s only full-service ...","omaha, ne",721189.0,3.0,• requires the ability to communicate effectiv...,full_time,68102,31055,Omaha,NE
8,2264355,bay west church,worship leader,it is an exciting time to be a part of our chu...,"palm bay, fl",28631247.0,5.0,"knowledge, skills and abilities: 1. proficient...",part_time,32905,12009,Palm Bay,FL
9,9615617,"glastender, inc.",inside customer service associate,glastender inc. is a family-owned manufacturer...,"saginaw, mi",1194336.0,4.0,the production supervisor must possess strong ...,full_time,48601,26145,Saginaw,MI


In [78]:
jobs.tail(50)

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state
115674,3906262691,"talentburst, an inc 5000 company",senior validation engineer,"title: validation engineer iii, req#: rocgjp00...","oceanside, ca",122451.0,1.0,NaN,contract,92054,06073,Oceanside,CA
115675,3906263017,realmanage family of brands,conveyance coordinator (operations) - (dtx2024...,"company overview: \n\nrealmanage, an inc. 5000...","dallas, tx",32705.0,4.0,NaN,full_time,75201,48113,Dallas,TX
115676,3906263085,idea public schools,assistant principal,company description idea public schools is a n...,"tampa, fl",81958.0,NaN,NaN,full_time,33602,12057,Tampa,FL
115677,3906263103,rose international,software engineer,date posted: 04/19/2024hiring organization: ro...,"mossville, il",12309.0,2.0,NaN,temporary,<NA>,NaN,NaN,IL
115678,3906263199,applicantz,cloud domain architect,this is a full time job with our fortune clien...,"houston, tx",24629.0,NaN,NaN,full_time,77002,48201,Houston,TX
115679,3906263301,synectics inc.,phlebotomist,description\n\nrepresents the face of our comp...,"irving, texas, united states",413796.0,4.0,NaN,contract,<NA>,NaN,NaN,TX
115680,3906263302,synectics inc.,phlebotomist,description\n\nrepresents the face of our comp...,"kansas city, missouri, united states",413796.0,4.0,NaN,contract,<NA>,NaN,NaN,KS
115681,3906263303,synectics inc.,phlebotomist,description\n\nrepresents the face of our comp...,"millbrook, alabama, united states",413796.0,4.0,NaN,contract,<NA>,NaN,NaN,AL
115682,3906263304,synectics inc.,phlebotomist,description\n\nrepresents the face of our comp...,"ridge, ks",413796.0,2.0,NaN,contract,67023,20035,Cambridge,KS
115683,3906263308,uams - university of arkansas for medical scie...,research assistant i- pathology,current university of arkansas system employee...,"little rock, ar",11250.0,4.0,NaN,full_time,72201,05119,Little Rock,AR


In [79]:
# We drop the rows where state is NaN even after our best efforts to extract it from the location string, we will not be able to use those rows in our analysis and 
# they are a small percentage of the total data so we can afford to drop them.

jobs = jobs.dropna(subset=['state'])
jobs.info()


<class 'pandas.DataFrame'>
Index: 111748 entries, 0 to 115723
Data columns (total 13 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   job_id        111748 non-null  str    
 1   company_name  110422 non-null  str    
 2   title         111748 non-null  str    
 3   description   111745 non-null  str    
 4   location      111748 non-null  str    
 5   company_id    110424 non-null  str    
 6   views         110298 non-null  float64
 7   skills_desc   2310 non-null    str    
 8   work_type     111748 non-null  str    
 9   zip_code      102977 non-null  string 
 10  fips          96434 non-null   str    
 11  city          99949 non-null   str    
 12  state         111748 non-null  str    
dtypes: float64(1), str(11), string(1)
memory usage: 11.9 MB


# Loading the student dataset and cleaning it

In [80]:
students = pd.read_csv('https://raw.githubusercontent.com/andrewmaina758/job-recommendation-system/refs/heads/main/students_dataset.csv')

# lets convert all the columns to lowercase to avoid any issues with case sensitivity when we do our analysis later on.
str_cols = students.select_dtypes(include=["object", "string"]).columns
students[str_cols] = students[str_cols].apply(lambda col: col.str.lower())

In [ ]:
# New data that we are using for bert embeddings, we will do the same cleaning as above to this data as well.
students_data = pd.read_csv("data\student_data.csv")
# lets convert all the columns to lowercase to avoid any issues with case sensitivity when we do our analysis later on.
str_cols = students_data.select_dtypes(include=["object", "string"]).columns
students_data[str_cols] = students_data[str_cols].apply(lambda col: col.str.lower())
students_data["Skill"] = (
    students_data["Skill"]
    .str.strip("[]")
    .str.replace('"', '')
)
students_data.head()

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\maina\AppData\Local\Temp\ipykernel_20108\4265789378.py:1: SyntaxWarning: invalid escape sequence '\s'
  students_data = pd.read_csv("data\student_data.csv")


,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree
0,stu000001,marketing,3.53,"client communication, market research, social ...",remote / flexible,boston university,sophomore,customer success,1.2,bachelor's
1,stu000002,finance,3.83,"collaboration, presentation skills, compliance...","seattle, wa",university of michigan,junior,risk management,1.6,bachelor's
2,stu000003,electrical engineering,3.70,"documentation, communication, verilog, problem...","washington, dc",university of pennsylvania,sophomore,electrical systems,0.9,bachelor's
3,stu000004,civil engineering,2.89,"project planning, civil 3d, attention to detai...","washington, dc",cornell university,senior,environmental engineering,3.2,bachelor's
4,stu000005,english,3.74,"editing, storytelling, content creation, prese...","san francisco bay area, ca",stanford university,freshman,communications,0.4,bachelor's


In [172]:
# Confirming that the data looks good and we have the skills column cleaned up as well.
students_data.iloc[900]

Student ID                                                            stu000901
Academic Major                                              information systems
GPA                                                                        3.41
Skill                         initiative, database management, time manageme...
Location Interest                                                   seattle, wa
University                                            san jose state university
School Year                                                              senior
Area of Experience                                            business analysis
Number of Experience (yrs)                                                  2.0
Degree                                                               bachelor's
Name: 900, dtype: object

In [81]:
students.head()


,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree
0,s0001,electrical engineering,3.35,"c++, embedded systems, matlab","new york, ny",university of michigan,sophomore,embedded systems,0.2,bachelors
1,s0002,data science,3.14,"machine learning, sql, statistics","detroit, mi",university of michigan,freshman,research,0.1,bachelors
2,s0003,computer science,3.40,"automation, data structures, linux, python","san francisco, ca",howard university,sophomore,machine learning,0.2,bachelors
3,s0004,economics,3.88,"data visualization, excel, research, sql, stat...","atlanta, ga",university of michigan,masters year 1,business analysis,2.5,masters
4,s0005,economics,3.36,"communication, data visualization, excel, rese...","san francisco, ca",georgia tech,phd year 2,business analysis,3.7,phd


In [173]:
students_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Student ID                  25000 non-null  str    
 1   Academic Major              25000 non-null  str    
 2   GPA                         25000 non-null  float64
 3   Skill                       25000 non-null  str    
 4   Location Interest           25000 non-null  str    
 5   University                  25000 non-null  str    
 6   School Year                 25000 non-null  str    
 7   Area of Experience          25000 non-null  str    
 8   Number of Experience (yrs)  25000 non-null  float64
 9   Degree                      25000 non-null  str    
dtypes: float64(2), str(8)
memory usage: 1.9 MB


# Text Vectorization

Before the system can say "this student matches this job", it must first convert job text into a numeric form. Our job dataset has text like title,description. Student dataset also has text like major, skills, experience. Computers do not compute meaning directly, so we convert text to numbers.

TF-IDF - is just a way to turn text into a vector. It gives higher importance to words that are; important in a document and not too common everywhere.
Cosine Similarity - once both the student profile and the job text are turned into vectors, cosine similarity answers how similar are this two pieces of text. High cosine similarity means the student profile looks more like the job text. This helps us to know which jobs to evn filter.

Why is this step important: Our dataset has many jobs. we do not want to score every single job with every feature immediately. Instead we do a fast filtering step:
- Convert jobs to vectors
- Converts student profile to vector
- Compare them
- Keep only the top jobs
We are doing candidate retrieval.

In [83]:
def build_job_text(jobs):
    jobs = jobs.copy()
    jobs["description"] = jobs["description"].fillna("")
    jobs["job_text"] = (jobs["title"] + " " + jobs["description"]).str.strip()
    return jobs

In [84]:
jobs_text = build_job_text(jobs)
jobs_text.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state,job_text
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021,Princeton,NJ,marketing coordinator job descriptiona leading...
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069,Fort Collins,CO,mental health therapist/counselor at aspen the...
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061,Cincinnati,OH,assitant restaurant manager the national exemp...
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059,New Hyde Park,NY,senior elder law / trusts and estates associat...
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057,Burlington,IA,service technician looking for hvac service te...


In [85]:
def normalize_job_title(title):
    import re
    import pandas as pd

    if pd.isna(title):
        return ""

    t = str(title).lower().strip()

    # replace separators with space
    t = re.sub(r"[/\\\-_(),]+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    # remove common modifiers
    noise_words = {
        "senior", "sr", "junior", "jr", "lead", "principal", "staff",
        "remote", "intern", "contract", "associate", "ii", "iii", "iv"
    }

    tokens = [tok for tok in t.split() if tok not in noise_words]
    t = " ".join(tokens)

    return t

In [86]:
jobs_text["title_normalized"] = jobs_text["title"].apply(normalize_job_title)

In [106]:
jobs_text[["title", "title_normalized"]].head(50)
jobs_text[["title", "title_normalized"]].tail(50)


,title,title_normalized
115674,senior validation engineer,validation engineer
115675,conveyance coordinator (operations) - (dtx2024...,conveyance coordinator operations dtx2024 6962
115676,assistant principal,assistant
115677,software engineer,software engineer
115678,cloud domain architect,cloud domain architect
115679,phlebotomist,phlebotomist
115680,phlebotomist,phlebotomist
115681,phlebotomist,phlebotomist
115682,phlebotomist,phlebotomist
115683,research assistant i- pathology,research assistant i pathology


In [88]:
jobs_text.head()

,job_id,company_name,title,description,location,company_id,views,skills_desc,work_type,zip_code,fips,city,state,job_text,title_normalized
0,921716,corcoran sawyer smith,marketing coordinator,job descriptiona leading real estate firm in n...,"princeton, nj",2774458.0,20.0,requirements: \n\nwe are seeking a college or ...,full_time,08540,34021,Princeton,NJ,marketing coordinator job descriptiona leading...,marketing coordinator
1,1829192,NaN,mental health therapist/counselor,"at aspen therapy and wellness , we are committ...","fort collins, co",NaN,1.0,NaN,full_time,80521,08069,Fort Collins,CO,mental health therapist/counselor at aspen the...,mental health therapist counselor
2,10998357,the national exemplar,assitant restaurant manager,the national exemplar is accepting application...,"cincinnati, oh",64896719.0,8.0,we are currently accepting resumes for foh - a...,full_time,45202,39061,Cincinnati,OH,assitant restaurant manager the national exemp...,assitant restaurant manager
3,23221523,"abrams fensterman, llp",senior elder law / trusts and estates associat...,senior associate attorney - elder law / trusts...,"new hyde park, ny",766262.0,16.0,this position requires a baseline understandin...,full_time,11040,36059,New Hyde Park,NY,senior elder law / trusts and estates associat...,elder law trusts and estates attorney
4,35982263,NaN,service technician,looking for hvac service tech with experience ...,"burlington, ia",NaN,3.0,NaN,full_time,52601,19057,Burlington,IA,service technician looking for hvac service te...,service technician


In [111]:
# next we tale the job_text and turn it into numbers
def vectorize_jobs(jobs_text):
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=10000,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
    )

    job_vectors = vectorizer.fit_transform(jobs_text["job_text"])
    return vectorizer, job_vectors

In [ ]:
# We will use bert embeddings instead of tf-idf for better performance and accuracy, 
# the code for tf-idf is left here for reference but we will not be using it in our analysis.
from sentence_transformers import SentenceTransformer

def vectorize_jobs_bert(jobs_text):
    # 1. Load a pre-trained model
    # 'all-MiniLM-L6-v2' is a great balance of speed and accuracy
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # 2. Encode the text
    # BERT converts each job description into a dense 384-dimensional vector
    new_job_vectors = model.encode(jobs_text["job_text"].tolist(), show_progress_bar=True)
    
    return model, new_job_vectors


c:\Users\maina\Documents\kdd-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Previous method using TF-IDF, we will not be using this in our analysis but we keep it here for reference.
vectorizer, job_vectors = vectorize_jobs(jobs_text)

In [ ]:
# New method using bert embeddings, this will give us better performance and accuracy when we do our analysis later on.
model, new_job_vectors = vectorize_jobs_bert(jobs_text)

HTTP Error 503 thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
c:\Users\maina\Documents\kdd-project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\maina\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/ap

In [113]:
print(f"Job vectors shape: {job_vectors.shape}")

Job vectors shape: (111748, 10000)


In [166]:
print(f"New job vectors shape: {new_job_vectors.shape}")

New job vectors shape: (111748, 384)


In [114]:
# we can quickly inspect the vocabulary
vectorizer.get_feature_names_out()[:20]

array(['aa', 'aaa', 'aap', 'ab', 'aba', 'abap', 'abb', 'abbott', 'abbvie',
       'abc', 'abenity', 'abercrombie', 'abet', 'abide', 'abides',
       'abilities', 'ability', 'abilityability', 'able', 'abnormal'],
      dtype=object)

In [ ]:
# Lets also inspect the first job vector to see what it looks like after we vectorize it using bert embeddings, 
# this will give us an idea of what the data looks like
print(new_job_vectors[0])

[-2.99145188e-02 -7.05849826e-02 -3.41785029e-02  6.76933601e-02
  2.36505885e-02  3.66323926e-02  5.92124574e-02 -1.16948979e-02
 -2.72486005e-02 -3.95866483e-02 -1.82558578e-02 -1.36507032e-02
  3.30789648e-02  6.73194751e-02  4.85873073e-02  2.73425039e-02
  8.02099928e-02 -7.52049731e-03 -6.41906559e-02 -5.89659670e-03
  2.79377797e-03 -3.48242708e-02 -5.45874052e-02 -3.57513018e-02
 -6.31982982e-02 -3.93862054e-02  3.49921137e-02 -9.49895568e-03
 -5.95280863e-02 -3.47071812e-02  3.44480947e-02 -3.90232056e-02
  8.92037898e-02  1.07532524e-01  8.32776278e-02  8.04968402e-02
 -3.08012534e-02 -1.81969181e-02  7.53299966e-02 -2.14589923e-03
 -1.11554032e-02  2.76455330e-03 -4.77034673e-02 -3.59378792e-02
  1.48405868e-03 -5.09461500e-02  7.77040375e-03  2.65066065e-02
 -1.18456483e-02  1.93885267e-02 -6.88615739e-02 -7.24537820e-02
 -4.64568883e-02 -1.13059823e-05 -1.76500585e-02  8.38923603e-02
 -1.68720577e-02 -8.39418545e-03  1.54265366e-03 -9.49955806e-02
  3.25840116e-02 -5.42178

Next we proceed to working on the student dataset and creating the vectors. We begin by asking ourselves which features are important for us. Considering we are doing text vectorization, columns which have text are more useful.

- Academic Major - This helps us define the student's academic direction. This makes us align closely to the program that the student is learning.
- Skill - Directly connects the student capability to the job requirements.
- Area of expertize - This gives more context than raw skills alone. Can distinguish between students with similar majors but different focus. 
- Degree and School year also have some weight as some job listings would indicate their preference for a candidate 

Simply saying: We first compare the student to jobs based on what they studied, what skills they have and what experience area they belong to.

In [93]:
def build_student_text(students):
    students = students.copy()
    students["student_text"] = (
        students["Academic Major"].fillna("") + " " +
        students["Skill"].fillna("") + " " +
        students["Area of Experience"].fillna("") + " " +
        students["School Year"].fillna("") + " "+
        students["Degree"].fillna("")
    ).str.strip()
    return students

In [ ]:
# Older code using TD-IDF
students_text = build_student_text(students)
students_text.head()
# students_text.iloc[97]

,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree,student_text
0,s0001,electrical engineering,3.35,"c++, embedded systems, matlab","new york, ny",university of michigan,sophomore,embedded systems,0.2,bachelors,"electrical engineering c++, embedded systems, ..."
1,s0002,data science,3.14,"machine learning, sql, statistics","detroit, mi",university of michigan,freshman,research,0.1,bachelors,"data science machine learning, sql, statistics..."
2,s0003,computer science,3.40,"automation, data structures, linux, python","san francisco, ca",howard university,sophomore,machine learning,0.2,bachelors,"computer science automation, data structures, ..."
3,s0004,economics,3.88,"data visualization, excel, research, sql, stat...","atlanta, ga",university of michigan,masters year 1,business analysis,2.5,masters,"economics data visualization, excel, research,..."
4,s0005,economics,3.36,"communication, data visualization, excel, rese...","san francisco, ca",georgia tech,phd year 2,business analysis,3.7,phd,"economics communication, data visualization, e..."


In [175]:
# New code using bert embeddings
new_student_text = build_student_text(students_data)
new_student_text.head()

,Student ID,Academic Major,GPA,Skill,Location Interest,University,School Year,Area of Experience,Number of Experience (yrs),Degree,student_text
0,stu000001,marketing,3.53,"client communication, market research, social ...",remote / flexible,boston university,sophomore,customer success,1.2,bachelor's,"marketing client communication, market researc..."
1,stu000002,finance,3.83,"collaboration, presentation skills, compliance...","seattle, wa",university of michigan,junior,risk management,1.6,bachelor's,"finance collaboration, presentation skills, co..."
2,stu000003,electrical engineering,3.70,"documentation, communication, verilog, problem...","washington, dc",university of pennsylvania,sophomore,electrical systems,0.9,bachelor's,"electrical engineering documentation, communic..."
3,stu000004,civil engineering,2.89,"project planning, civil 3d, attention to detai...","washington, dc",cornell university,senior,environmental engineering,3.2,bachelor's,"civil engineering project planning, civil 3d, ..."
4,stu000005,english,3.74,"editing, storytelling, content creation, prese...","san francisco bay area, ca",stanford university,freshman,communications,0.4,bachelor's,"english editing, storytelling, content creatio..."


The vectorizer defines the "language space". Jobs define the vocabulary, students must be projected into that same vocabulary. If we trained a separate vectorizer on students, comparisons would be meaningless.

In [95]:
# Since we already trained a vectorizer on the job data, we can reuse that same vectorizer to transform the student text into vectors. 
# This way we ensure that the job and student vectors are in the same feature space and can be compared directly.

student_vectors = vectorizer.transform(students_text["student_text"])
print(f"Student vectors shape: {student_vectors.shape}")

Student vectors shape: (600, 10000)


In [177]:
# We use the bert model we trained on the job data, we can reuse that same model to encode the student text into vectors.
new_student_vectors = model.encode(
    new_student_text["student_text"].tolist(),
    show_progress_bar=True
)
print(new_student_vectors.shape)

Batches: 100%|██████████| 782/782 [02:20<00:00,  5.57it/s]


(25000, 384)


In [ ]:
# This was the older code using TD-IDF.
# new_student_vectors = vectorizer.transform(vector_text["student_text"])
# print(f"New student vectors shape: {new_student_vectors.shape}")

New student vectors shape: (25000, 10000)


So far the question becomes, for one student which jobs are most similar?
We take one student vector and compare it against all job vectors. That gives one similarity score per job. Then we sort from highest to lowest and keep the top ones. This is the first retriever layer of our recommender

In [ ]:
# Older Version using TD-IDF
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(new_student_vectors[900], job_vectors).flatten()

top_k = 100
top_idx = similarities.argsort()[::-1][:top_k]

top_jobs = jobs_text.iloc[top_idx][["job_id", "title_normalized", "location", "state"]].copy()
top_jobs["similarity"] = similarities[top_idx]

print("Top retrieved postings:")
print(top_jobs.head(10))

Top retrieved postings:
           job_id                 title_normalized  \
67341  3902872004                       bi analyst   
53566  3901944374               power bi architect   
40432  3900074564               power bi developer   
30829  3894984328               power bi architect   
69191  3903186267               power bi developer   
52961  3901941624               power bi developer   
39770  3899538009               power bi developer   
5937   3885113214      data analyst power bi & sql   
53558  3901944307  business intelligence developer   
54506  3901951800               power bi developer   

                                  location state  similarity  
67341                           irvine, ca    CA    0.379415  
53566                      jersey city, nj    NJ    0.365107  
40432  district of columbia, united states    DC    0.364535  
30829            new jersey, united states    NJ    0.359604  
69191                            tampa, fl    FL    0.356208  
529

In [ ]:
# Newer version using bert embeddings.
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(new_student_vectors, new_job_vectors)
print(similarity_matrix.shape)

top_k = 5
top_matches = np.argsort(-similarity_matrix, axis=1)[:, :top_k]

results = []

for student_idx, job_idxs in enumerate(top_matches):
    for rank, job_idx in enumerate(job_idxs, start=1):
        results.append({
            "student_index": student_idx,
            "student_text": new_student_text.iloc[student_idx]["student_text"],
            "job_index": job_idx,
            "job_text": jobs_text.iloc[job_idx]["job_text"],
            "similarity_score": similarity_matrix[student_idx, job_idx],
            "rank": rank
        })

matches_df = pd.DataFrame(results)
matches_df.head(10)

In [156]:
top_jobs.head(20)

,job_id,title_normalized,location,state,similarity
67341,3902872004,bi analyst,"irvine, ca",CA,0.379415
53566,3901944374,power bi architect,"jersey city, nj",NJ,0.365107
40432,3900074564,power bi developer,"district of columbia, united states",DC,0.364535
30829,3894984328,power bi architect,"new jersey, united states",NJ,0.359604
69191,3903186267,power bi developer,"tampa, fl",FL,0.356208
52961,3901941624,power bi developer,"dallas, tx",TX,0.350807
39770,3899538009,power bi developer,"merrifield, va",VA,0.350268
5937,3885113214,data analyst power bi & sql,"san francisco, ca",CA,0.345557
53558,3901944307,business intelligence developer,"milwaukee, wi",WI,0.344537
54506,3901951800,power bi developer,"dallas, tx",TX,0.342306


Right now as we notice, our retrieval returns individual rows. This means we see a row after the other based on the similarity score. This way a student just gets to see repeated titles. However what we want to show them is that:
- X is a strong match
- Y is also a strong match
- Z is another possible direction
So grouping by title helps us move from posting-level retrieval to role-level recommendation.

**What happens under the hood?**
For one student, we compute similarity against all job postings, we take the matching postings, group those postings by title, summarize each and finally rank the titles.

In [157]:
# summarize retrieved postings into matched titles
# ensure title_normalized exists in top_jobs before grouping
if "title_normalized" not in top_jobs.columns:
    top_jobs = top_jobs.merge(
        jobs_text[["job_id", "title_normalized"]].drop_duplicates(),
        on="job_id",
        how="left"
    )

#top_jobs["title_normalized"] = top_jobs["title_normalized"].fillna(top_jobs["title"])

title_scores = (
    top_jobs.groupby("title_normalized", as_index=False)
    .agg(
        avg_similarity=("similarity", "mean"),
        retrieved_posting_count=("job_id", "count")
    )
    .rename(columns={"title_normalized": "title"})
    .sort_values("avg_similarity", ascending=False)
)

print("\nMatched titles:")
print(title_scores.head(10))


Matched titles:
                                     title  avg_similarity  \
3                               bi analyst        0.379415   
10             data analyst power bi & sql        0.345557   
45                        powerbi designer        0.342072   
43                       powerbi architect        0.341476   
38           power bi developer only on w2        0.339588   
34                          power bi admin        0.338429   
35                      power bi architect        0.335824   
18  etl business intelligence bi developer        0.331722   
36                      power bi developer        0.325873   
50                           sql developer        0.324696   

    retrieved_posting_count  
3                         1  
10                        1  
45                        1  
43                        1  
38                        1  
34                        1  
35                        5  
18                        1  
36                       12  


In [158]:

title_scores["combined_score"] = (
    title_scores["avg_similarity"] *
    np.log1p(title_scores["retrieved_posting_count"])
)

title_scores = title_scores.sort_values("combined_score", ascending=False)

In [159]:
# we keep the best matched titles
top_titles = title_scores.head()["title"].tolist()

print("\nTop matched titles:")
print(top_titles)


Top matched titles:
['power bi developer', 'power bi architect', 'business intelligence developer', 'business intelligence analyst', 'sql dba']


# Location interest matching

In [160]:
def normalize_state_value(x):
    import pandas as pd

    if pd.isna(x):
        return ""

    x = str(x).strip().lower()
    
    if len(x) == 2:
        return x.upper()

    return STATE_MAP.get(x, x.upper())

In [163]:
# we keep only top 5 demand states per title
top_titles_df = title_scores.head(5).copy()

student_row = students_data.iloc[900]
preferred_location_raw = student_row["Location Interest"]
preferred_state = normalize_state_value(preferred_location_raw)

print(f"Student: {student_row['Academic Major']} student")
print(f"Preferred location: {preferred_location_raw}\n")
print("Top Career Matches:\n")

for i, (_, row) in enumerate(top_titles_df.iterrows(), start=1):
    title = row["title"]
    fit_score = row["avg_similarity"]

    all_states = (
        jobs_text[jobs_text["title_normalized"] == title]
        .groupby("state", as_index=False)
        .size()
        .rename(columns={"size": "demand_count"})
        .sort_values("demand_count", ascending=False)
    )

    top_states = all_states.head(5)

    # Convert states to readable string
    state_list = [
        f"{r['state']} ({r['demand_count']})"
        for _, r in top_states.iterrows()
    ]

    # Check if preferred state exists
    # preferred_exists = preferred_state in all_states["state"].values

    print(f"{i}. {title.title()}")
    print(f"   Fit Score: {fit_score:.2f}")
    print(f"   Top Demand Locations: {', '.join(state_list)}")

    # if preferred_exists:
    #     print("   ✓ Your preferred location is available for this role")
    # else:
    #     print("   ✗ Not aligned with your preferred location")

    print()

Student: information systems student
Preferred location: seattle, wa

Top Career Matches:

1. Power Bi Developer
   Fit Score: 0.33
   Top Demand Locations: TX (4), CA (2), NJ (2), DC (1), CT (1)

2. Power Bi Architect
   Fit Score: 0.34
   Top Demand Locations: NJ (4), FL (1)

3. Business Intelligence Developer
   Fit Score: 0.31
   Top Demand Locations: PA (4), CA (3), GA (3), MA (2), OH (2)

4. Business Intelligence Analyst
   Fit Score: 0.29
   Top Demand Locations: CA (4), TX (4), GA (2), FL (1), NJ (1)

5. Sql Dba
   Fit Score: 0.28
   Top Demand Locations: TX (2), CT (1), AZ (1), IL (1), NH (1)



# Machine Learning Model

We ask ourselves this question, what level of opportunity is realistic for this student right now?
We consider students who fall in different categories:
- Freshmen
- Juniors
- Masters
- PHD

We try to address the issue of students being matched for roles they are not even qualified for.
We would train a model to predict student readiness level for jobs such as:
- Internship
- Entry level
- Junior
- Advanced/research

Then we can use that prediction to filter/weight recommended jobs
A student readiness level can be estimated from the following fields
- School year
- Degree
- GPA 
- Number of experience
- Area of experience
- Skill
- Academic major

The first question we ask ourselves is what are the labels. In our case, we will use the labels below
- internship - freshmen/sophomore. Experience years are very low, low GPA
- entry_level
- Junior
- advanced_research

